# 02 · Parse NYC CityGML

**Goal:** extract buildings, labeled surfaces, polygons, and metadata into ordinary Python records. The XML machinery now lives in a reusable reader; this notebook focuses on how to use its output.

Run from top to bottom with the repository's `.venv` Python interpreter. In VS Code use **Select Kernel → Python Environments → .venv**. Alternatively run `uv run jupyter lab` from the repository root and select the project environment.

Our roadmap: Inspect → **Parse** → Transform → Analyze → Visualize. No mesh conversion or solar calculation happens here.

In [1]:
from pathlib import Path
from collections import Counter
from dataclasses import asdict
from contextlib import closing
import pandas as pd
from IPython.display import display, Markdown
from gismo_2026_fall_hackathon.citygml import iter_buildings

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists() and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
GML_FILE = ROOT / "data/DA_WISE_GML/DA_WISE_GMLs/DA1_3D_Buildings_Merged.gml"
SAMPLE_SIZE = 100
assert GML_FILE.is_file(), f"Missing local dataset: {GML_FILE}"
print(f"Sample: up to {SAMPLE_SIZE} top-level buildings from {GML_FILE.name}")

Sample: up to 100 top-level buildings from DA1_3D_Buildings_Merged.gml


## 1. One call replaces the XML plumbing

`iter_buildings(path, limit=None)` yields one top-level `Building` at a time. `limit` counts buildings, not their nested parts; zero yields nothing. Leaving it as `None` reads the entire file. Here we collect a bounded sample so we can inspect it repeatedly.

Processed XML members are released as the reader proceeds. Retaining the returned records still costs memory: collecting every building in NYC into a list would defeat the purpose of streaming. If stopping iteration early, use `contextlib.closing` to close the source file promptly, as shown below.

In [2]:
with closing(iter_buildings(GML_FILE, limit=SAMPLE_SIZE)) as reader:
    buildings = list(reader)
assert buildings, "No buildings were extracted"
print(f"Extracted {len(buildings)} top-level buildings")

Extracted 100 top-level buildings


## 2. Understand the returned records

- **Building:** `id`, `name`, `kind`, `metadata`, `surfaces`, nested `parts`, and `issues`.
- **Surface:** `id`, `kind` (such as RoofSurface), `owner_path`, `metadata`, and `polygons`.
- **Polygon:** `id` if declared, `exterior` (an N×3 NumPy array), `interiors` (a list of hole arrays), `crs`, and `dimension`.
- **InspectionIssue:** a `context` identifier and a readable `message`.

BuildingPart records use the same Building type and retain their own surfaces and metadata. `owner_path` traces the owning building and parts. Absent source IDs remain `None`; diagnostic paths use a missing-ID label. Metadata IDs remain strings, preserving any leading zeros.

The helper below walks this returned hierarchy; it does not parse XML.

In [3]:
def walk_buildings(records):
    for building in records:
        yield building
        yield from walk_buildings(building.parts)

owners = list(walk_buildings(buildings))
building_table = pd.DataFrame([
    {"id": b.id, "name": b.name, "kind": b.kind,
     "surfaces": len(b.surfaces), "parts": len(b.parts), "issues": len(b.issues),
     **{f"metadata.{k}": v for k, v in b.metadata.items()}}
    for b in owners
])
display(building_table.head(10))
first = buildings[0]
print("First building:", first.id)
print("Metadata:", first.metadata)
print("Building parts:", len(first.parts))

,id,name,kind,surfaces,parts,issues,metadata.DOITT_ID,metadata.BIN,metadata.SOURCE_ID
0,gml_ZL7GWAR67EC4S4ILQ40LOJB7Y8NM02D0HAWS,Bldg_1210018481,Building,8,0,0,294532,4465953,1210018481
1,gml_VHKIINWDOF6WB4EAYEUNKTM4WOAJO2TUFXZN,Bldg_1210015304,Building,11,0,0,428189,4540322,1210015304
2,gml_A6J4L0RIDQ1932MHI20F9XRU3YDRKT44WAIL,Bldg_1210017488,Building,15,0,0,482372,4467298,1210017488
3,gml_YJR00O3LS8E92KBJZV8SLHZ0N9C10ZXBKUJM,Bldg_1210002406,Building,14,0,0,299105,4453836,1210002406
4,gml_A5PXVHPLMVNN56TZAT3PEHBRCSFVQPWGMMEF,Bldg_1210018263,Building,20,0,0,198536,4467050,1210018263
5,gml_KTJS0DRDK2YKF5UV62LUEVX3TXLC5M6ZG7N4,Bldg_1210012044,Building,14,0,0,720932,4458757,1210012044
6,gml_0IBFF0VI8POAMOOP0J78NRGVQ8S1KA4AFZQ3,Bldg_1210013982,Building,13,0,0,600596,4466419,1210013982
7,gml_FLHMT60V609N5DXCYS378XIIM5Z7KMKAM4W9,Bldg_1210013584,Building,10,0,0,493209,4466107,1210013584
8,gml_ZTWN5H02N27AN49S4LYUOD3VR5GTJWM5EYZL,Bldg_1210013969,Building,22,0,0,44768,4465567,1210013969
9,gml_76IHR59I9ENHEE34HTMFLNS65ZVV907RWVHW,Bldg_1210007279,Building,12,0,0,461214,4466211,1210007279


First building: gml_ZL7GWAR67EC4S4ILQ40LOJB7Y8NM02D0HAWS
Metadata: {'DOITT_ID': '294532', 'BIN': '4465953', 'SOURCE_ID': '1210018481'}
Building parts: 0


## 3. Surface and polygon inventory

A single semantic surface may contain multiple polygons. A polygon can have holes, so one polygon is not necessarily one ring. We preserve that grouping for the mesh-conversion step rather than flattening everything into unrelated vertices.

In [4]:
surfaces = [s for b in owners for s in b.surfaces]
surface_table = pd.DataFrame([
    {"surface_id": s.id, "kind": s.kind, "owner_path": " / ".join(s.owner_path),
     "polygons": len(s.polygons), "interior_rings": sum(len(p.interiors) for p in s.polygons)}
    for s in surfaces
], columns=["surface_id", "kind", "owner_path", "polygons", "interior_rings"])
display(surface_table.head(12))
display(surface_table.groupby("kind").agg(surfaces=("kind", "size"),
                                         polygons=("polygons", "sum"),
                                         interior_rings=("interior_rings", "sum")))
polygons = [p for s in surfaces for p in s.polygons]
print(f"{len(surfaces)} semantic surfaces; {len(polygons)} valid polygons")
print("CRS declarations:", dict(Counter(p.crs for p in polygons)))

,surface_id,kind,owner_path,polygons,interior_rings
0,gml_G73EI5D84XNG6BTW7RBQKARYXFGFBMWRIZET,RoofSurface,gml_ZL7GWAR67EC4S4ILQ40LOJB7Y8NM02D0HAWS,1,0
1,gml_AWDTFCCAUHPYIQ209FIFPFQDY87RB58G6AVL,GroundSurface,gml_ZL7GWAR67EC4S4ILQ40LOJB7Y8NM02D0HAWS,1,0
2,gml_NSLH7QDQZR6UT6SCR8E5FA4GFU9ZII2I5RMQ,WallSurface,gml_ZL7GWAR67EC4S4ILQ40LOJB7Y8NM02D0HAWS,1,0
3,gml_4AI8CANHCM8FM5Z760FNXECMROHQZXZCTCHQ,WallSurface,gml_ZL7GWAR67EC4S4ILQ40LOJB7Y8NM02D0HAWS,1,0
4,gml_0F3FXXE2W4MMT4MEM82Z85RY26SOX5VKYIH0,WallSurface,gml_ZL7GWAR67EC4S4ILQ40LOJB7Y8NM02D0HAWS,1,0
5,gml_790AAJ0WB474LV0UKSS5YEM0F0E0JCYLXWGM,WallSurface,gml_ZL7GWAR67EC4S4ILQ40LOJB7Y8NM02D0HAWS,1,0
6,gml_I2ZYF4W88XCCFBAF0LB2X94JF9N8FWGZONMR,WallSurface,gml_ZL7GWAR67EC4S4ILQ40LOJB7Y8NM02D0HAWS,1,0
7,gml_23Z3SO1RXCUICSULUNZI7PTGRX2MQNY2YSN7,WallSurface,gml_ZL7GWAR67EC4S4ILQ40LOJB7Y8NM02D0HAWS,1,0
8,gml_DQAU2Q1M1FLWJXDU7AWB4PV3MMZVVMROF07H,RoofSurface,gml_VHKIINWDOF6WB4EAYEUNKTM4WOAJO2TUFXZN,1,0
9,gml_DAU42ADRDOA0Z8WFAYSX4HHQ0F0OY6QS05Q6,RoofSurface,gml_VHKIINWDOF6WB4EAYEUNKTM4WOAJO2TUFXZN,1,0


,surfaces,polygons,interior_rings
kind,,,
GroundSurface,100,100,0
RoofSurface,223,223,0
WallSurface,1119,1119,0


1442 semantic surfaces; 1442 valid polygons
CRS declarations: {'EPSG:2263': 1442}


## 4. Access a roof's coordinates

These XYZ coordinates have not been reprojected, triangulated, repaired, or converted into physical measurements. EPSG:2263 specifies horizontal US survey feet; Z units and vertical datum are still unverified. Do not use this array to calculate a physical slope or area until units are reconciled.

The exterior includes the repeated closing vertex. Interior boundaries stay separate. The following example selects the first successfully extracted roof polygon, with an explanatory message if the sample has none.

In [5]:
roof = next((s for s in surfaces if s.kind == "RoofSurface" and s.polygons), None)
if roof is None:
    print("No valid roof polygon in this sample; inspect the issues below or choose another sample.")
else:
    polygon = roof.polygons[0]
    print("Roof:", roof.id, "| owner:", roof.owner_path)
    print("Surface metadata:", roof.metadata)
    print("Polygon ID:", polygon.id, "| CRS:", polygon.crs, "| dimension:", polygon.dimension)
    display(pd.DataFrame(polygon.exterior, columns=["X", "Y", "Z"]))
    print("Interior rings:", len(polygon.interiors))
    for index, hole in enumerate(polygon.interiors, 1):
        print(f"Hole {index}: {len(hole)} positions")

Roof: gml_G73EI5D84XNG6BTW7RBQKARYXFGFBMWRIZET | owner: ('gml_ZL7GWAR67EC4S4ILQ40LOJB7Y8NM02D0HAWS',)
Surface metadata: {'DOITT_ID': '294532', 'BIN': '4465953', 'SOURCE_ID': '1210018481'}
Polygon ID: None | CRS: EPSG:2263 | dimension: 3


,X,Y,Z
0,1.007684e+06,143034.642787,24.6758
1,1.007631e+06,143048.987574,24.6758
2,1.007630e+06,143045.848801,24.6758
3,1.007625e+06,143026.412160,24.6758
4,1.007624e+06,143023.884606,24.6758
5,1.007677e+06,143009.539818,24.6758
6,1.007684e+06,143034.642787,24.6758


Interior rings: 0


## 5. Check extraction issues

The reader supports CityGML 2.0 inline Polygon / LinearRing / posList geometry under labeled roof, wall, ground, and related boundary surfaces. It validates XYZ grouping, finite values, distinct vertices, closure, and consistent CRS declarations within each polygon. A bad hole excludes its entire polygon so that extraction does not accidentally fill the hole.

Unsupported coordinate encodings, geometry references, and other unsupported geometry are reported. References are **not resolved** in this stage. A surface can therefore exist with no usable polygons. Issues belong to the owning building or part; check both. Malformed XML and an unsupported document version stop iteration with a clear error.

This is extraction validation, not a check of polygon planarity, self-intersections, outward normals, or mesh topology.

In [6]:
issues = [dict(owner_id=b.id, **asdict(issue)) for b in owners for issue in b.issues]
if issues:
    issue_table = pd.DataFrame(issues)
    display(issue_table.head(20))
    display(issue_table.value_counts("message").rename("count").to_frame())
else:
    print("No extraction issues in this sample. This does not establish citywide validity.")
print(f"Extraction totals: {len(buildings)} buildings, {len(owners) - len(buildings)} parts, "
      f"{len(surfaces)} surfaces, {len(polygons)} valid polygons, "
      f"{sum(len(p.interiors) for p in polygons)} holes, {len(issues)} issues")

# Small sanity checks on the returned data, rather than on XML details.
assert len(buildings) <= SAMPLE_SIZE
assert all(p.dimension == 3 and p.exterior.shape[1] == 3 for p in polygons)
assert all(s.owner_path for s in surfaces)

No extraction issues in this sample. This does not establish citywide validity.
Extraction totals: 100 buildings, 0 parts, 1442 surfaces, 1442 valid polygons, 0 holes, 0 issues


## What comes next?

Stage 1 showed the source XML and labeled outlines. Stage 2 gives us reusable, structured geometry with the same source IDs and surface meanings. You can now work with `building.surfaces` and `surface.polygons` instead of XML tags.

For **Stage 3**, we will choose a small sample, verify vertical units, and convert polygons (including holes) into usable mesh faces while keeping their building and surface identities. Geometry repair, triangulation, solar scoring, and exports remain outside this notebook.

The reader's synthetic tests cover cases not necessarily present in this sample. Run them from the repository root with:

```sh
uv run python -m unittest discover -s tests -v
```